# Практика · 06. Чому повнозвʼязна мережа не годиться> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) · ДЗ: [homework.html](homework.html)Тут ми не переказуємо лекцію, а **міряємо** те, що в ній стверджується.Що зробимо:1. згенеруємо датасет фігур 28 × 28 формулами — коло, квадрат, ромб;2. побудуємо дві мережі: повнозвʼязну і згорткову;3. порахуємо їхні параметри **на папері** і звіримо з `sum(p.numel() for p in model.parameters())`;4. навчимо обидві на предметах по центру й перевіримо на **зсунутих** — це головний дослід теми;5. побудуємо криву точності проти величини зсуву;6. перемішаємо пікселі однією перестановкою й побачимо, що повнозвʼязна мережа цього   не помічає, а згорткова — помічає;7. доведемо це без жодного навчання: переставимо стовпці ваг і покажемо, що вихід   мережі **не змінився ні на біт**.> ⚠️ **Мережа не потрібна:** датасет ми малюємо самі, формулами. Нічого не завантажується.> ⏱ Зошит навчає чотири мережі. Заміряно: близько трьох хвилин на чотирьох ядрах без відеокарти.

In [ ]:
# Усе, що знадобиться. torch тут потрібен саме тому, що тема про мережі.
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Зерна фіксуємо на самому початку: без них числа в тебе й у лекції розійдуться.
torch.manual_seed(0)
torch.set_num_threads(4)

print("numpy  ", np.__version__)
print("torch  ", torch.__version__)
print("пристрій:", "CPU (GPU нам тут і не потрібен)")

## 1 · Датасет: три фігури, намальовані формуламиКожне зображення — квадрат 28 × 28 чисел від 0 до 1. Фігура світла (0.9), тло темне (0.1),зверху трохи гаусового шуму, щоб задача не була зовсім тепличною.Класів три: **коло**, **квадрат**, **ромб**. Три класи означають, що випадкове вгадуваннядає точність приблизно **0.333** — це наш нуль відліку, з яким ми весь час порівнюємо.Головний параметр генератора — `max_shift`: наскільки центр фігури може відхилитись відцентру полотна. Саме ним ми потім і зробимо головний дослід.

In [ ]:
SIZE = 28
CLASS_NAMES = ["коло", "квадрат", "ромб"]


def draw_shape(kind, center_x, center_y, radius, rng):
    """Одна фігура на полотні 28 × 28. Значення від 0 до 1."""
    # rows і cols — координати кожного пікселя; так формула пишеться без жодного циклу
    rows, cols = np.mgrid[0:SIZE, 0:SIZE]
    dx = cols - center_x
    dy = rows - center_y

    if kind == 0:                                   # коло: відстань до центру
        inside = dx * dx + dy * dy <= radius * radius
    elif kind == 1:                                 # квадрат: обидві координати малі
        inside = (np.abs(dx) <= radius * 0.88) & (np.abs(dy) <= radius * 0.88)
    else:                                           # ромб: сума модулів координат мала
        inside = np.abs(dx) + np.abs(dy) <= radius * 1.25

    image = np.where(inside, 0.9, 0.1)
    # шум додаємо навмисно: без нього мережа вивчила б задачу за дві епохи
    image = image + rng.normal(0, 0.08, (SIZE, SIZE))
    return np.clip(image, 0, 1).astype(np.float32)


def make_dataset(count, max_shift, seed):
    """count зображень; центр фігури гуляє в межах ±max_shift пікселів."""
    rng = np.random.default_rng(seed)
    images = np.zeros((count, 1, SIZE, SIZE), dtype=np.float32)
    labels = np.zeros(count, dtype=np.int64)
    for i in range(count):
        kind = int(rng.integers(0, 3))
        shift_x = int(rng.integers(-max_shift, max_shift + 1))
        shift_y = int(rng.integers(-max_shift, max_shift + 1))
        radius = float(rng.uniform(5.0, 6.5))
        images[i, 0] = draw_shape(kind, 13.5 + shift_x, 13.5 + shift_y, radius, rng)
        labels[i] = kind
    return torch.from_numpy(images), torch.from_numpy(labels)


start = time.time()
# навчальна вибірка: предмети майже по центру, гуляють лише на ±1 піксель
train_x, train_y = make_dataset(1200, max_shift=1, seed=42)
# перший тест: той самий розподіл
test_x, test_y = make_dataset(300, max_shift=1, seed=7)
# другий тест: ті самі фігури, але зсунуті на ±6 пікселів
shift_x_data, shift_y_data = make_dataset(300, max_shift=6, seed=11)

print(f"згенеровано за {time.time() - start:.1f} с")
print("навчальна вибірка:", tuple(train_x.shape), "мітки:", tuple(train_y.shape))
print("тест той самий розподіл:", tuple(test_x.shape))
print("тест зсунуті предмети:  ", tuple(shift_x_data.shape))
print("класів у навчальній вибірці:", torch.bincount(train_y).tolist())

Подивимось на дані очима. Верхній ряд — навчальні приклади (усе по центру), нижній —зсунуті, на яких ми будемо перевіряти. Форма та сама, клас той самий, змінилось лише місце.

In [ ]:
figure, axes = plt.subplots(2, 6, figsize=(11, 4))
for column in range(6):
    axes[0, column].imshow(train_x[column, 0], cmap="gray", vmin=0, vmax=1)
    axes[0, column].set_title(CLASS_NAMES[train_y[column]], fontsize=10)
    axes[1, column].imshow(shift_x_data[column, 0], cmap="gray", vmin=0, vmax=1)
    axes[1, column].set_title(CLASS_NAMES[shift_y_data[column]], fontsize=10)
for ax in axes.ravel():
    ax.axis("off")
axes[0, 0].set_ylabel("навчання")
figure.suptitle("угорі — навчальні (по центру), унизу — перевірка на зсунутих", fontsize=11)
plt.tight_layout()
plt.show()
print("перший ряд — те, на чому мережа вчиться; другий — те, на чому ми її зловимо")

## 2 · Повнозвʼязна мережа і підрахунок параметрів на паперіМережа найпростіша з можливих:* `Flatten` — сітка 28 × 28 перетворюється на рядок із 784 чисел;* `Linear(784, 64)` + `ReLU` — прихований шар;* `Linear(64, 3)` — три класи.Порахуймо параметри руками, перш ніж питати про це PyTorch. У повнозвʼязного шару`Linear(a, b)` рівно `a × b` ваг плюс `b` зсувів.

In [ ]:
def build_mlp():
    """Повнозвʼязна мережа: жодних припущень про те, що вхід — зображення."""
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 64),
        nn.ReLU(),
        nn.Linear(64, 3),
    )


# ── рахуємо на папері ──
mlp_layer1 = 784 * 64 + 64          # ваги першого шару плюс його зсуви
mlp_layer2 = 64 * 3 + 3             # ваги вихідного шару плюс його зсуви
mlp_by_hand = mlp_layer1 + mlp_layer2

torch.manual_seed(0)
mlp = build_mlp()
mlp_by_torch = sum(p.numel() for p in mlp.parameters())

print(f"перший шар  784 × 64 + 64 = {mlp_layer1}")
print(f"другий шар   64 ×  3 +  3 = {mlp_layer2}")
print(f"разом на папері           = {mlp_by_hand}")
print(f"sum(p.numel() ...)        = {mlp_by_torch}")
assert mlp_by_hand == mlp_by_torch, "ручний підрахунок розійшовся з PyTorch!"
print("✅ збігається")

## 3 · Згорткова мережа і той самий підрахунокТут інша логіка. У згорткового шару `Conv2d(вхідних каналів, вихідних каналів, k)`параметрів `k × k × вхідних × вихідних + вихідних` — і **розміру зображення в цій формулінемає**. Це і є головна відмінність.Між згортками стоїть `MaxPool2d(2)` — огрублення сітки вдвічі (з кожного квадратика 2 × 2лишається найбільше число). Параметрів у неї нуль; докладно вона розібрана в темі 08, тутми користуємось нею як інструментом, щоб довести розмір карти до 3 × 3.

In [ ]:
def build_cnn():
    """Згорткова мережа: локальність, спільні ваги, зменшення розмірності."""
    return nn.Sequential(
        nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 28 -> 14
        nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 14 -> 7
        nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 7  -> 3
        nn.Flatten(),
        nn.Linear(16 * 3 * 3, 3),
    )


# ── рахуємо на папері ──
conv1 = 3 * 3 * 1 * 8 + 8            # 8 ядер 3×3 над одним каналом
conv2 = 3 * 3 * 8 * 16 + 16          # 16 ядер 3×3 над вісьмома каналами
conv3 = 3 * 3 * 16 * 16 + 16         # ще 16 ядер над шістнадцятьма каналами
head = 16 * 3 * 3 * 3 + 3            # повнозвʼязна голова над 144 числами
cnn_by_hand = conv1 + conv2 + conv3 + head

torch.manual_seed(0)
cnn = build_cnn()
cnn_by_torch = sum(p.numel() for p in cnn.parameters())

print(f"conv1  3×3×1×8  + 8  = {conv1}")
print(f"conv2  3×3×8×16 + 16 = {conv2}")
print(f"conv3  3×3×16×16+ 16 = {conv3}")
print(f"голова 144×3    + 3  = {head}")
print(f"разом на папері      = {cnn_by_hand}")
print(f"sum(p.numel() ...)   = {cnn_by_torch}")
assert cnn_by_hand == cnn_by_torch, "ручний підрахунок розійшовся з PyTorch!"
print("✅ збігається")
print()
print(f"у повнозвʼязної мережі параметрів у {mlp_by_torch / cnn_by_torch:.1f} раза більше")

Перевіримо, що розмір карти справді доходить до 3 × 3 — це те саме «зменшення розмірності»,через яке повнозвʼязна голова коштує 435 параметрів, а не мільйони.

In [ ]:
sample = train_x[:1]                       # один приклад, щоб подивитись форму на кожному кроці
print("вхід:", tuple(sample.shape))
for layer in cnn:
    sample = layer(sample)
    print(f"{layer.__class__.__name__:<12} -> {tuple(sample.shape)}")

## 4 · Навчання і головний дослідОбидві мережі вчаться на **тих самих** даних: предмети майже по центру, зсув до ±1 пікселя.Перевіряємо двічі — на тому самому розподілі й на зсунутих предметах.

In [ ]:
def train(model, images, labels, watch=None, epochs=12):
    """Звичайний цикл навчання. Зерно всередині — щоб числа повторювались.

    watch — пара (зображення, мітки) для перевірки після кожної епохи. Якщо її
    передати, функція поверне список точностей по епохах; це знадобиться,
    щоб порівняти не тільки кінцевий результат, а й дорогу до нього.
    """
    torch.manual_seed(0)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_function = nn.CrossEntropyLoss()
    history = []
    for epoch in range(epochs):
        order = torch.randperm(len(images))     # свій порядок прикладів у кожній епосі
        for start_index in range(0, len(images), 64):
            batch = order[start_index:start_index + 64]
            optimizer.zero_grad()
            loss = loss_function(model(images[batch]), labels[batch])
            loss.backward()
            optimizer.step()
        if watch is not None:
            history.append(round(accuracy(model, watch[0], watch[1]), 3))
    return history


def accuracy(model, images, labels):
    """Частка правильних відповідей. Без градієнтів — це просто перевірка."""
    with torch.no_grad():
        predicted = model(images).argmax(1)
    return (predicted == labels).float().mean().item()


torch.manual_seed(0)
mlp = build_mlp()
start = time.time()
# watch дає нам точність після кожної епохи — знадобиться в розділі про перемішування
mlp_history = train(mlp, train_x, train_y, watch=(test_x, test_y))
mlp_seconds = time.time() - start

mlp_same = accuracy(mlp, test_x, test_y)
mlp_shifted = accuracy(mlp, shift_x_data, shift_y_data)
print(f"MLP навчався {mlp_seconds:.1f} с")
print(f"точність по епохах:  {mlp_history}")
print(f"той самий розподіл: {mlp_same:.3f}")
print(f"зсунуті предмети:   {mlp_shifted:.3f}")

In [ ]:
torch.manual_seed(0)
cnn = build_cnn()
start = time.time()
cnn_history = train(cnn, train_x, train_y, watch=(test_x, test_y))
cnn_seconds = time.time() - start

cnn_same = accuracy(cnn, test_x, test_y)
cnn_shifted = accuracy(cnn, shift_x_data, shift_y_data)
print(f"CNN навчалась {cnn_seconds:.1f} с")
print(f"точність по епохах:  {cnn_history}")
print(f"той самий розподіл: {cnn_same:.3f}")
print(f"зсунуті предмети:   {cnn_shifted:.3f}")

In [ ]:
print(f"{'модель':<8}{'параметрів':>12}{'той самий':>12}{'зсунуті':>10}")
print("-" * 42)
print(f"{'MLP':<8}{mlp_by_torch:>12}{mlp_same:>12.3f}{mlp_shifted:>10.3f}")
print(f"{'CNN':<8}{cnn_by_torch:>12}{cnn_same:>12.3f}{cnn_shifted:>10.3f}")
print("-" * 42)
print("рівень випадкового вгадування для трьох класів: 0.333")
print()
print("Читай так: обидві мережі розвʼязали задачу «як учили» — і саме тому")
print("перевірка на зсунутих і потрібна. Без неї ми б написали у звіті, що все чудово.")

## 5 · Крива: як точність залежить від величини зсувуОдин рядок таблиці мало що каже. Побудуємо криву: для кожного `max_shift` від 0 до 8згенеруємо свій тестовий набір і поміряємо обидві мережі. Навчати нікого не треба —моделі вже готові.

In [ ]:
shift_values = list(range(9))
mlp_curve, cnn_curve = [], []
for shift in shift_values:
    curve_x, curve_y = make_dataset(300, max_shift=shift, seed=11)
    mlp_curve.append(accuracy(mlp, curve_x, curve_y))
    cnn_curve.append(accuracy(cnn, curve_x, curve_y))

print(f"{'зсув':>5}{'MLP':>9}{'CNN':>9}")
for shift in shift_values:
    print(f"{shift:>5}{mlp_curve[shift]:>9.3f}{cnn_curve[shift]:>9.3f}")

In [ ]:
plt.figure(figsize=(8, 4.2))
plt.plot(shift_values, mlp_curve, "o-", label="повнозвʼязна (MLP)")
plt.plot(shift_values, cnn_curve, "s-", label="згорткова (CNN)")
plt.axhline(1 / 3, linestyle="--", color="gray", label="випадкове вгадування")
plt.xlabel("максимальний зсув предмета, пікселів")
plt.ylabel("точність")
plt.ylim(0, 1.05)
plt.legend()
plt.grid(alpha=0.3)
plt.title("Що робить зсув із двома мережами")
plt.show()
print("CNN тримається довше — але й вона сповзає. Терпимість часткова, а не повна.")

## 6 · Перемішування пікселів: доказ, що MLP не бачить зображенняВізьмемо **одну** випадкову перестановку 784 позицій і застосуємо її до **всіх**зображень — і навчальних, і тестових. Людина після цього не впізнає нічого.

In [ ]:
permutation_rng = np.random.default_rng(2024)
pixel_permutation = permutation_rng.permutation(784)


def shuffle_pixels(images):
    """Та сама перестановка для всіх зображень набору."""
    flat = images.reshape(len(images), -1)          # (N, 784)
    return flat[:, pixel_permutation].reshape(len(images), 1, SIZE, SIZE)


train_x_shuffled = shuffle_pixels(train_x)
test_x_shuffled = shuffle_pixels(test_x)

figure, axes = plt.subplots(1, 6, figsize=(11, 2.2))
for column in range(3):
    axes[column].imshow(test_x[column, 0], cmap="gray", vmin=0, vmax=1)
    axes[column].set_title(CLASS_NAMES[test_y[column]], fontsize=10)
    axes[column + 3].imshow(test_x_shuffled[column, 0], cmap="gray", vmin=0, vmax=1)
    axes[column + 3].set_title("перемішано", fontsize=10)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()
print("перші три — оригінали, останні три — вони ж після перестановки")

### Спершу доказ без жодного навчанняТвердження лекції було таке: для будь-якої повнозвʼязної мережі на звичайних даних існує**точно так само хороша** мережа на перемішаних — з тими самими вагами, лише переставленимитією самою перестановкою.Це можна не вірити на слово, а перевірити за секунду. Візьмемо вже навчену мережу,переставимо стовпці матриці ваг першого шару — і подамо їй перемішані зображення.Якщо твердження правильне, виходи збіжаться **до останнього знака**.

In [ ]:
import copy

twin = copy.deepcopy(mlp)
with torch.no_grad():
    # twin[1] — це Linear(784, 64); його ваги мають форму (64, 784),
    # тому переставляємо саме стовпці, тобто другий вимір
    twin[1].weight.copy_(mlp[1].weight[:, pixel_permutation])

with torch.no_grad():
    original_output = mlp(test_x)
    twin_output = twin(test_x_shuffled)

difference = (original_output - twin_output).abs().max().item()
print(f"максимальна різниця виходів: {difference:.3e}")
assert torch.allclose(original_output, twin_output, atol=1e-5), "виходи розійшлись!"
print("✅ мережі-близнюки дають той самий результат")
print()
print("Отже, перемішування пікселів не забирає в повнозвʼязної мережі рівно нічого:")
print("вона й на початкових даних не користувалась тим, які пікселі сусідні.")

### А тепер навчимо обидві мережі з нуля на перемішаних данихДоказ вище стосувався готової мережі. Перевіримо, що й навчання з нуля дає те саме —і подивимось, що станеться зі згортковою.

In [ ]:
torch.manual_seed(0)
mlp_shuffled = build_mlp()
mlp_shuffled_history = train(mlp_shuffled, train_x_shuffled, train_y,
                             watch=(test_x_shuffled, test_y))
mlp_perm_accuracy = accuracy(mlp_shuffled, test_x_shuffled, test_y)
print(f"MLP на перемішаних: {mlp_perm_accuracy:.3f}   (на звичайних було {mlp_same:.3f})")
print()
print("Порівняй не кінцеві числа, а всю дорогу — епоха за епохою:")
print(f"{'епоха':>7}{'звичайні':>11}{'перемішані':>13}")
for epoch_index in range(len(mlp_history)):
    print(f"{epoch_index + 1:>7}{mlp_history[epoch_index]:>11.3f}"
          f"{mlp_shuffled_history[epoch_index]:>13.3f}")

In [ ]:
torch.manual_seed(0)
cnn_shuffled = build_cnn()
cnn_shuffled_history = train(cnn_shuffled, train_x_shuffled, train_y,
                             watch=(test_x_shuffled, test_y))
cnn_perm_accuracy = accuracy(cnn_shuffled, test_x_shuffled, test_y)
print(f"CNN на перемішаних: {cnn_perm_accuracy:.3f}   (на звичайних було {cnn_same:.3f})")
print()
print(f"{'епоха':>7}{'звичайні':>11}{'перемішані':>13}")
for epoch_index in range(len(cnn_history)):
    print(f"{epoch_index + 1:>7}{cnn_history[epoch_index]:>11.3f}"
          f"{cnn_shuffled_history[epoch_index]:>13.3f}")
print()
print("Згорткова просіла, але не розвалилась: на 1200 прикладах їй вистачає")
print("ємності вивчити навіть кришиво. Дивись, скільки епох на це пішло.")

In [ ]:
print(f"{'модель':<8}{'звичайні':>12}{'перемішані':>13}{'втрата':>10}")
print("-" * 44)
print(f"{'MLP':<8}{mlp_same:>12.3f}{mlp_perm_accuracy:>13.3f}{mlp_same - mlp_perm_accuracy:>10.3f}")
print(f"{'CNN':<8}{cnn_same:>12.3f}{cnn_perm_accuracy:>13.3f}{cnn_same - cnn_perm_accuracy:>10.3f}")
print("-" * 44)
print("Повнозвʼязна мережа не помітила, що зображення знищили.")
print("Згорткова помітила одразу: у неї відібрали те, на чому вона стоїть — сусідство пікселів.")

## 7 · Що ми заміряли| Твердження лекції | Чим підтверджено тут ||---|---|| у повнозвʼязної мережі параметрів набагато більше | ручний підрахунок і `sum(p.numel())` збіглися для обох мереж || розгортання руйнує сусідство | перестановка стовпців ваг дає **побітово** той самий вихід || MLP не бачить зображення як зображення | навчання з нуля на перемішаних даних дає ту саму точність || CNN спирається саме на сусідство | та сама перестановка ламає їй точність || ознака, вивчена в одному місці, не переноситься | падіння точності при зсуві предмета || згортка дає лише **часткову** терпимість до зсуву | крива CNN теж сповзає, просто повільніше |---## Завдання### 🟢 Рівень 1 — БазаДодай у генератор четвертий клас — **кільце** (коло з дірою всередині: `radius` зовні,`radius * 0.55` усередині). Пересобери датасет на чотири класи, перенавчи обидві мережі йпобудуй ту саму таблицю.**Зроблено, якщо:** у таблиці чотири колонки чисел і ти назвав новий рівень випадковоговгадування (для чотирьох класів це вже не 0.333).### 🟡 Рівень 2 — ПлюсНавчи повнозвʼязну мережу **на зсунутих** предметах (`max_shift=6`) і перевір її на такихсамих зсунутих. Порівняй із результатом навчання по центру.**Зроблено, якщо:** ти показав числом, що MLP цілком здатна вивчити задачу зі зсувом — алетільки якщо їй показати кожне положення, і сказав, у скільки разів для цього знадобилосьбільше даних або епох.### 🔴 Рівень 3 — ВикликЗаміряй, як швидко ламається згорткова мережа, якщо перемішувати не всі пікселі, а частку.Зроби перестановку, яка міняє місцями лише `p` відсотків позицій (`p` = 0, 10, 25, 50, 100),навчи CNN на кожному варіанті й побудуй криву точності проти `p`.**Зроблено, якщо:** є графік із пʼятьма точками і письмовий висновок про те, чи падінняплавне, чи різке — і чому саме таке.### Підказки* Кільце найпростіше зробити як різницю двох умов: `inside_outer & ~inside_inner`.* Щоб перемішати лише частку позицій, візьми випадкову підмножину індексів і переставляй  усередині неї, лишивши решту на місці.* Для рівня 3 зменш кількість епох до 8 — крива від цього не зміниться, а чекати буде  вдвічі менше.